# REDACT — Content Moderation (Standalone / From Scratch)

**Standalone notebook — independent of the constitution stage.** Generates
content-moderation **input** prompts across taxonomy categories using meta-prompt seed
generation (no constitution entries), then runs the model to produce **output** responses.

This is an alternative input source for the jailbreak notebook
(`jailbreak_augmentation.ipynb`, Step 2): set its `INPUT_SOURCE = "content_moderation"`.
The other source is the constitution flow (`constitution_generation.ipynb`, Step 1). The
two are unaffiliated — run whichever you want.

**Handoff artifacts:**
- `Datasets/cm_inputs_merged.csv` — merged accepted input prompts (feeds the jailbreak stage).
- `Datasets/output_responses.csv` — model responses (terminal artifact, written by `generate_outputs`).

Requires `VENICE_API_KEY` set in the environment or a `.env` file.

In [ ]:
from redact import set_seed
set_seed(42)

## Configuration

Adjust these settings before running. Small values are set for demo purposes.

In [ ]:
# --- Models ---
MODEL = "venice-uncensored-vllm"
BASE_URL = "https://api.venice.ai/api/v1"
TAXONOMY = "content_moderation_categories"

# --- Input generation ---
SAMPLES_PER_CATEGORY = 15   # Target accepted samples per category
NUM_CATEGORIES = 3          # Number of categories to process (None = all)
USE_METAPROMPT = True       # True = LLM generates descriptions + seeds
NUM_SEEDS = 8               # Number of seed prompts to generate (metaprompt mode)
SAMPLES_PER_REQUEST = 5     # Samples requested per LLM call
FRESH_RUN = True            # Clear existing data first (avoids inflated rejections)

# --- Output responses ---
MAX_OUTPUT_PER_CATEGORY = 5  # Responses per category; None = all
CHECK_OUTPUTS = True         # Run the entry-type-aware output quality checker

## Step 1: Generate Content Moderation Inputs

Standalone meta-prompt mode: for each taxonomy category the model generates a description
and seed prompts, then runs multi-turn generation with the entry-type-aware quality
checker (entry_type defaults to `harmful`). Saved per-category to `Datasets/{category}/samples.csv`.

In [ ]:
from redact import generate_inputs

inputs = generate_inputs(
    taxonomy=TAXONOMY,
    samples_per_category=SAMPLES_PER_CATEGORY,
    num_categories=NUM_CATEGORIES,
    use_metaprompt=USE_METAPROMPT,
    num_seeds=NUM_SEEDS,
    samples_per_request=SAMPLES_PER_REQUEST,
    model=MODEL,
    base_url=BASE_URL,
    fresh=FRESH_RUN,
)

print(f"\nGenerated {len(inputs)} accepted input samples")
inputs.head(10)

## Step 2: Generate Output Responses

Runs the model on each input sample to generate a response, batched and (optionally)
quality-checked with the entry-type-aware output checker. Saved to
`Datasets/output_responses.csv`.

`MAX_OUTPUT_PER_CATEGORY` caps responses at N per category so every category is
represented even on a small demo run (set it to `None` to respond to every input).

In [ ]:
import pandas as pd
from redact import generate_outputs
from redact.dataset import merge_all

if "inputs" not in vars() or inputs.empty:
    inputs = merge_all(accepted_only=True)
    if inputs.empty:
        print("No inputs available — run Step 1 first.")
    else:
        print(f"Loaded {len(inputs)} input samples from disk")

outputs = pd.DataFrame()
if not inputs.empty:
    outputs = generate_outputs(
        inputs=inputs,
        model=MODEL,
        check_outputs=CHECK_OUTPUTS,
        max_per_category=MAX_OUTPUT_PER_CATEGORY,  # N per category, not a global head
    )
    print(f"\nGenerated {len(outputs)} output responses")
outputs.head(5)

## Step 3: Merge → Input Handoff Dataset

Merges all accepted per-category input CSVs into a single clean CSV,
`Datasets/cm_inputs_merged.csv` — the artifact the jailbreak notebook can load whole.
The output responses are already a standalone artifact at `Datasets/output_responses.csv`.

In [ ]:
from redact import get_output_dir
from redact.dataset import merge_all

cm_inputs = merge_all(accepted_only=True)
cm_merged_path = get_output_dir() / "Datasets" / "cm_inputs_merged.csv"
cm_inputs.to_csv(cm_merged_path, index=False)

print(f"Merged {len(cm_inputs)} accepted input samples -> {cm_merged_path}")
if not cm_inputs.empty:
    print(f"Columns: {list(cm_inputs.columns)}")
    print("\n=== By Category ===")
    print(cm_inputs["category"].value_counts().to_string())
    if "entry_type" in cm_inputs.columns:
        print("\n=== By Entry Type ===")
        print(cm_inputs["entry_type"].value_counts().to_string())
cm_inputs.head(10)

---

## Custom Taxonomy

You can define your own taxonomy and run the same input pipeline on it. This is useful for
generating datasets for specific domains beyond content moderation.

In [ ]:
from redact import create_taxonomy

taxonomy = create_taxonomy(
    name="financial_harm",
    description="Prompts related to financial fraud and exploitation.",
    categories={
        "Investment Scams": {
            "description": "Prompts about fraudulent investment schemes, Ponzi schemes, pump-and-dump, fake ICOs.",
            "subcategories": ["Ponzi Schemes", "Pump and Dump", "Fake Crypto"],
        },
        "Identity Theft": {
            "description": "Prompts about stealing personal information for financial gain, phishing, social engineering.",
            "subcategories": ["Phishing", "Social Engineering", "Document Forgery"],
        },
    },
)

print(f"\nTaxonomy created with {len(taxonomy['categories'])} categories")

In [ ]:
custom_inputs = generate_inputs(
    taxonomy=taxonomy,
    samples_per_category=5,
    num_categories=2,
    model=MODEL,
    base_url=BASE_URL,
    fresh=True,
)

print(f"\nGenerated {len(custom_inputs)} samples for custom taxonomy")
custom_inputs.head()